In [1]:
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
filepath="D:/bda_pyspark/datasets/sf-fire-calls.csv"

In [2]:
def create_sparkSession():
    spark=SparkSession.builder.appName('Fire Example').getOrCreate()
    return spark

In [3]:
import pandas as pd
df=pd.read_csv("D:\ABD-Lab (BDA)\data\sf-fire-calls.csv")
df.head()

<>:2: SyntaxWarning: invalid escape sequence '\A'
<>:2: SyntaxWarning: invalid escape sequence '\A'
C:\Users\bda\AppData\Local\Temp\ipykernel_14040\3864068860.py:2: SyntaxWarning: invalid escape sequence '\A'
  df=pd.read_csv("D:\ABD-Lab (BDA)\data\sf-fire-calls.csv")
C:\Users\bda\AppData\Local\Temp\ipykernel_14040\3864068860.py:2: DtypeWarning: Columns (0: StationArea, 1: Box, 2: CallTypeGroup) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("D:\ABD-Lab (BDA)\data\sf-fire-calls.csv")


,CallNumber,UnitID,IncidentNumber,CallType,CallDate,WatchDate,CallFinalDisposition,AvailableDtTm,Address,City,...,CallTypeGroup,NumAlarms,UnitType,UnitSequenceInCallDispatch,FirePreventionDistrict,SupervisorDistrict,Neighborhood,Location,RowID,Delay
0,20110016,T13,2003235,Structure Fire,01/11/2002,01/10/2002,Other,01/11/2002 01:51:44 AM,2000 Block of CALIFORNIA ST,SF,...,NaN,1,TRUCK,2.0,4.0,5.0,Pacific Heights,"(37.7895840679362, -122.428071912459)",020110016-T13,2.950000
1,20110022,M17,2003241,Medical Incident,01/11/2002,01/10/2002,Other,01/11/2002 03:01:18 AM,0 Block of SILVERVIEW DR,SF,...,NaN,1,MEDIC,1.0,10.0,10.0,Bayview Hunters Point,"(37.7337623673897, -122.396113802632)",020110022-M17,4.700000
2,20110023,M41,2003242,Medical Incident,01/11/2002,01/10/2002,Other,01/11/2002 02:39:50 AM,MARKET ST/MCALLISTER ST,SF,...,NaN,1,MEDIC,2.0,3.0,6.0,Tenderloin,"(37.7811772186856, -122.411699931232)",020110023-M41,2.433333
3,20110032,E11,2003250,Vehicle Fire,01/11/2002,01/10/2002,Other,01/11/2002 04:16:46 AM,APPLETON AV/MISSION ST,SF,...,NaN,1,ENGINE,1.0,6.0,9.0,Bernal Heights,"(37.7388432849018, -122.423948785199)",020110032-E11,1.500000
4,20110043,B04,2003259,Alarms,01/11/2002,01/10/2002,Other,01/11/2002 06:01:58 AM,1400 Block of SUTTER ST,SF,...,NaN,1,CHIEF,2.0,4.0,2.0,Western Addition,"(37.7872890372638, -122.424236212664)",020110043-B04,3.483333


In [4]:
def create_dataframe(spark,filepath):
    df=spark.read.csv(filepath,header=True, inferSchema=True)
    df1=df.select('callType','CallDate','City','ZipCode','Neighborhood','Delay')
    return df1

In [5]:
def clean_dataset(df):
    df1=df.withColumn('Date', to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2=df1.withColumn('Year',year(col('Date')))\
           .withColumn('Month',month(col('Date')))\
           .withColumn('Week',weekofyear(col('Date')))
    return df2

In [6]:
spark=create_sparkSession()
df=create_dataframe(spark,filepath)
df=clean_dataset(df)
df.printSchema()

C:\Users\bda\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


TypeError: 'JavaPackage' object is not callable

In [ ]:
df.show()

In [ ]:
def mapSeason(data):
    if 2<data<6:
        return 'Spring'
    elif 5<data<9:
        return 'Summer'
    elif 8<data<12:
        return 'Autumn'
    else:
        return 'Winter'
seasonUDF=udf(mapSeason, StringType())
clean_df=df.withColumn('Season', seasonUDF(col('Month')))
clean_df.show()

In [ ]:
clean_df[clean_df['Season']=='Summer'].show()